<a href="https://colab.research.google.com/github/Sineth7077/project-aura/blob/main/notebooks/01_environment_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Aura 2.0 - Notebook 01: Environment SetupThis notebook initializes the Google Colab environment for Project Aura 2.0:- Installs required Python dependencies (ultralytics, mediapipe, opencv-python-headless,  numpy, pillow, tqdm, torch, torchvision, tensorflow)- Avoids unnecessarily reinstalling already-compatible PyTorch and TensorFlow versions- Clones the GitHub repository if not already present- Adds /content/project-aura to sys.path before importing src- Detects and reports Colab, Python, GPU, CUDA, and installed libraries- Verifies imports from src.core.config and src.core.constants- Verifies load_config() reports runtime_type and gpu_enabled correctly- Prints a clear environment verification summarySafe to rerun: all steps are idempotent.

In [ ]:
# ==========================================================================# Step 0: Install required dependencies# ==========================================================================import subprocessimport sysdef _is_installed(package):    """Check if a package is already installed."""    try:        __import__(package)        return True    except ImportError:        return FalseREQUIRED_PACKAGES = [    "numpy",    "PIL",          # pillow    "tqdm",    "ultralytics",    "mediapipe",    "cv2",          # opencv-python-headless]# Install core packages that are missingmissing = [p for p in REQUIRED_PACKAGES if not _is_installed(p)]if missing:    print(f"Installing missing packages: {missing}")    !pip install --quiet {' '.join(missing)}else:    print("All core packages already installed.")# PyTorch and torchvision - check version compatibility before installingTORCH_INSTALLED = _is_installed("torch")TORCHVISION_INSTALLED = _is_installed("torchvision")if not TORCH_INSTALLED or not TORCHVISION_INSTALLED:    print("Installing PyTorch and torchvision (CPU build)...")    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpuelse:    import torch    print(f"PyTorch already installed: {torch.__version__} - skipping reinstall.")# TensorFlow - check if already installed before installingTF_INSTALLED = _is_installed("tensorflow")if not TF_INSTALLED:    print("Installing TensorFlow...")    !pip install --quiet tensorflowelse:    import tensorflow as tf    print(f"TensorFlow already installed: {tf.__version__} - skipping reinstall.")print("Dependency check complete.")

In [ ]:
import sys
import subprocess
import importlib.util

def is_installed(module_name):
    return importlib.util.find_spec(module_name) is not None

# Map Python import names to pip package names
packages = {
    "numpy": "numpy",
    "PIL": "pillow",
    "tqdm": "tqdm",
    "ultralytics": "ultralytics",
    "mediapipe": "mediapipe",
    "cv2": "opencv-python-headless",
}

missing = [
    package
    for module, package in packages.items()
    if not is_installed(module)
]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", *missing
    ])
else:
    print("All core vision packages are installed.")

# Preserve existing PyTorch installation
if not is_installed("torch"):
    print("Installing PyTorch and TorchVision...")
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "torch", "torchvision"
    ])
else:
    import torch
    print("PyTorch already installed:", torch.__version__)

# Preserve existing TensorFlow installation
if not is_installed("tensorflow"):
    print("Installing TensorFlow...")
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "tensorflow"
    ])
else:
    import tensorflow as tf
    print("TensorFlow already installed:", tf.__version__)

print("\nDependency check complete.")

In [ ]:
# ==========================================================================# Step 1: Clone the GitHub repository if not already present# ==========================================================================import osimport sysPROJECT_DIR = "/content/project-aura"REPO_URL = "https://github.com/Sineth7077/project-aura.git"if not os.path.exists(PROJECT_DIR):    print(f"Cloning {REPO_URL} ...")    !git clone {REPO_URL} {PROJECT_DIR}else:    print(f"Project directory already exists: {PROJECT_DIR}")    !git -C {PROJECT_DIR} fetch origin    !git -C {PROJECT_DIR} checkout main    !git -C {PROJECT_DIR} pull origin mainprint("Repository ready")

In [ ]:
import os
import subprocess

PROJECT_DIR = "/content/project-aura"
REPO_URL = "https://github.com/Sineth7077/project-aura.git"

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    if os.path.exists(PROJECT_DIR):
        raise RuntimeError(
            f"{PROJECT_DIR} exists but is not a Git repository. "
            "Inspect it before proceeding."
        )

    print("Cloning Project Aura...")
    subprocess.run(
        ["git", "clone", REPO_URL, PROJECT_DIR],
        check=True
    )
else:
    print("Project repository already exists.")
    print("Using the existing checkout without overwriting local changes.")

print("Repository:", PROJECT_DIR)
print("Repository ready.")

In [ ]:
# ==========================================================================# Step 2: Add project to Python path# ==========================================================================if PROJECT_DIR not in sys.path:    sys.path.insert(0, PROJECT_DIR)print(f"Python path updated: {PROJECT_DIR} in sys.path = {PROJECT_DIR in sys.path}")

In [ ]:
import sys
import importlib

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

importlib.invalidate_caches()

print("Project directory:", PROJECT_DIR)
print("Project in Python path:", PROJECT_DIR in sys.path)
print("src folder exists:", os.path.isdir(
    os.path.join(PROJECT_DIR, "src")
))

In [ ]:
# ==========================================================================# Step 3: Detect and report environment# ==========================================================================import platformimport importlibtry:    from google.colab import drive as _gdrive    IN_COLAB = Trueexcept ImportError:    IN_COLAB = FalseCOLAB_GPU = os.getenv("COLAB_GPU", "").lower() in ("true", "1")COLAB_RELEASE_TAG = os.getenv("COLAB_RELEASE_TAG", "")PYTHON_VERSION = sys.version# PyTorch GPU detectionTORCH_AVAILABLE = FalseTORCH_VERSION = NoneTORCH_CUDA_AVAILABLE = FalseTORCH_CUDA_DEVICE_NAME = Nonetry:    import torch    TORCH_AVAILABLE = True    TORCH_VERSION = torch.__version__    TORCH_CUDA_AVAILABLE = bool(torch.cuda.is_available())    if TORCH_CUDA_AVAILABLE:        TORCH_CUDA_DEVICE_NAME = torch.cuda.get_device_name(0)except ImportError:    pass# TensorFlow GPU detectionTENSORFLOW_AVAILABLE = FalseTENSORFLOW_VERSION = NoneTF_GPU_AVAILABLE = Falsetry:    import tensorflow as tf    TENSORFLOW_AVAILABLE = True    TENSORFLOW_VERSION = tf.__version__    TF_GPU_AVAILABLE = len(tf.config.list_physical_devices('GPU')) > 0except ImportError:    pass# Other library versionsdef _version(module_name):    try:        mod = importlib.import_module(module_name)        return getattr(mod, "__version__", "unknown")    except ImportError:        return NoneNUMPY_VERSION = _version('numpy')PILLOW_VERSION = _version('PIL')MEDIAPIPE_VERSION = _version('mediapipe')OPENCV_VERSION = _version('cv2')ULTRALYTICS_VERSION = _version('ultralytics')TORCHVISION_VERSION = _version('torchvision')print("=== Environment Detection ===")print(f"Running in Colab: {IN_COLAB}")print(f"COLAB_GPU env var: {COLAB_GPU}")print(f"COLAB_RELEASE_TAG env var: {COLAB_RELEASE_TAG or "(not set)"}")print(f"Python version: {PYTHON_VERSION.split()[0]}")print(f"Platform: {platform.platform()}")print(f"PyTorch available: {TORCH_AVAILABLE} (v{TORCH_VERSION})")print(f"PyTorch CUDA available: {TORCH_CUDA_AVAILABLE}")if TORCH_CUDA_DEVICE_NAME:    print(f"  GPU device: {TORCH_CUDA_DEVICE_NAME}")print(f"TensorFlow available: {TENSORFLOW_AVAILABLE} (v{TENSORFLOW_VERSION})")print(f"TensorFlow GPU available: {TF_GPU_AVAILABLE}")print(f"NumPy: {NUMPY_VERSION}")print(f"Pillow: {PILLOW_VERSION}")print(f"MediaPipe: {MEDIAPIPE_VERSION}")print(f"OpenCV: {OPENCV_VERSION}")print(f"Ultralytics: {ULTRALYTICS_VERSION}")print(f"TorchVision: {TORCHVISION_VERSION}")

In [ ]:
import os
import sys
import platform
import importlib

# Detect Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = (
        bool(os.getenv("COLAB_RELEASE_TAG"))
        or os.getenv("COLAB_GPU", "").lower() in ("true", "1")
    )

# Detect PyTorch GPU
TORCH_AVAILABLE = False
TORCH_CUDA_AVAILABLE = False
TORCH_GPU_NAME = None
TORCH_VERSION = None

try:
    import torch

    TORCH_AVAILABLE = True
    TORCH_VERSION = torch.__version__
    TORCH_CUDA_AVAILABLE = torch.cuda.is_available()

    if TORCH_CUDA_AVAILABLE:
        TORCH_GPU_NAME = torch.cuda.get_device_name(0)

except ImportError:
    pass

# Detect TensorFlow GPU
TENSORFLOW_AVAILABLE = False
TF_GPU_AVAILABLE = False
TENSORFLOW_VERSION = None

try:
    import tensorflow as tf

    TENSORFLOW_AVAILABLE = True
    TENSORFLOW_VERSION = tf.__version__

    TF_GPU_AVAILABLE = (
        len(tf.config.list_physical_devices("GPU")) > 0
    )

except ImportError:
    pass

GPU_AVAILABLE = TORCH_CUDA_AVAILABLE or TF_GPU_AVAILABLE

print("========== PROJECT AURA ENVIRONMENT ==========")
print("Colab:", IN_COLAB)
print("Colab release:", os.getenv("COLAB_RELEASE_TAG", "Not detected"))
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

print("\n========== GPU STATUS ==========")
print("PyTorch:", TORCH_VERSION)
print("PyTorch CUDA:", TORCH_CUDA_AVAILABLE)
print("TensorFlow:", TENSORFLOW_VERSION)
print("TensorFlow GPU:", TF_GPU_AVAILABLE)
print("GPU available:", GPU_AVAILABLE)

if TORCH_GPU_NAME:
    print("GPU device:", TORCH_GPU_NAME)

In [ ]:
# ==========================================================================# Step 4: Verify imports from src.core.config and src.core.constants# ==========================================================================from src.core.config import (    ColabConfig,    AuraConfig,    load_config,    detect_gpu,    detect_tpu,)from src.core.constants import (    PROJECT_NAME,    PROJECT_VERSION,    SUPPORTED_VISION_MODELS,)config = load_config()print("\n=== Import Verification ===")print("src.core.config: OK")print("src.core.constants: OK")print(f"Project: {PROJECT_NAME} v{PROJECT_VERSION}")print(f"Supported vision models: {SUPPORTED_VISION_MODELS}")print(f"Config loaded: debug={config.debug}, log_level={config.log_level}")print(f"Colab runtime_type: {config.colab.runtime_type}")print(f"Colab gpu_enabled: {config.colab.gpu_enabled}")print(f"Colab tpu_enabled: {config.colab.tpu_enabled}")

In [ ]:
from src.core.config import (
    ColabConfig,
    AuraConfig,
    load_config,
    detect_gpu,
    detect_tpu,
)

from src.core.constants import (
    PROJECT_NAME,
    PROJECT_VERSION,
    SUPPORTED_VISION_MODELS,
)

config = load_config()

print("========== PROJECT IMPORT VERIFICATION ==========")

print("Configuration imports: SUCCESS")
print("Constants imports: SUCCESS")

print("Project:", PROJECT_NAME)
print("Version:", PROJECT_VERSION)

print("Supported vision models:", SUPPORTED_VISION_MODELS)

print("\nConfiguration:")
print("Runtime:", config.colab.runtime_type)
print("GPU enabled:", config.colab.gpu_enabled)
print("TPU enabled:", config.colab.tpu_enabled)
print("Debug:", config.debug)
print("Log level:", config.log_level)

In [ ]:
# ==========================================================================# Step 5: Verify load_config() reports correctly on Colab# ==========================================================================print("\n=== Colab Configuration Verification ===")print(f"IN_COLAB: {IN_COLAB}")print(f"config.colab.runtime_type: {config.colab.runtime_type}")print(f"config.colab.gpu_enabled: {config.colab.gpu_enabled}")print(f"config.detect_gpu(): {detect_gpu()}")# On Colab with T4 GPU, runtime_type should be 'colab' and gpu_enabled should be Trueif IN_COLAB:    expected_runtime = "colab"    expected_gpu = TORCH_CUDA_AVAILABLE or TF_GPU_AVAILABLE    runtime_ok = config.colab.runtime_type == expected_runtime    gpu_ok = config.colab.gpu_enabled == expected_gpu    print(f"Expected runtime_type: {expected_runtime} - {"OK" if runtime_ok else "MISMATCH"}")    print(f"Expected gpu_enabled: {expected_gpu} - {"OK" if gpu_ok else "MISMATCH"}")    if runtime_ok and gpu_ok:        print("Configuration verification PASSED for Colab environment.")    else:        print("Configuration verification FAILED - see mismatches above.")else:    print("Not running in Colab - configuration reflects local/production environment.")

In [ ]:
print("========== CONFIGURATION VERIFICATION ==========")

print("Running in Colab:", IN_COLAB)
print("Runtime detected:", config.colab.runtime_type)
print("GPU detected by config:", detect_gpu())
print("GPU enabled in config:", config.colab.gpu_enabled)

if IN_COLAB:
    runtime_ok = config.colab.runtime_type == "colab"
    gpu_ok = config.colab.gpu_enabled == GPU_AVAILABLE

    print("\nRuntime check:", "PASS" if runtime_ok else "FAIL")
    print("GPU configuration check:", "PASS" if gpu_ok else "FAIL")

    if not runtime_ok:
        print("WARNING: Colab runtime was not detected correctly.")

    if not gpu_ok:
        print("WARNING: GPU configuration does not match hardware detection.")

    if runtime_ok and gpu_ok:
        print("\nColab configuration verification PASSED.")
else:
    print("Local/production environment detected.")

In [ ]:
# ==========================================================================# Step 6: Print environment verification summary# ==========================================================================print("\n" + "=" * 60)print("PROJECT AURA 2.0 - ENVIRONMENT VERIFICATION SUMMARY")print("=" * 60)print(f"Runtime: {"Google Colab" if IN_COLAB else "Local/Production"}")print(f"Python: {PYTHON_VERSION.split()[0]}")print(f"PyTorch: {TORCH_VERSION or "not installed"} (CUDA: {TORCH_CUDA_AVAILABLE})")print(f"TensorFlow: {TENSORFLOW_VERSION or "not installed"} (GPU: {TF_GPU_AVAILABLE})")print(f"GPU detected by config.detect_gpu(): {detect_gpu()}")print(f"Config gpu_enabled: {config.colab.gpu_enabled}")print(f"NumPy: {NUMPY_VERSION}")print(f"Pillow: {PILLOW_VERSION}")print(f"MediaPipe: {MEDIAPIPE_VERSION}")print(f"OpenCV: {OPENCV_VERSION}")print(f"Ultralytics: {ULTRALYTICS_VERSION}")print(f"TorchVision: {TORCHVISION_VERSION}")print(f"Project directory: {PROJECT_DIR}")print(f"Project in sys.path: {PROJECT_DIR in sys.path}")print("=" * 60)if IN_COLAB and not TORCH_CUDA_AVAILABLE and not TF_GPU_AVAILABLE:    print("\nNOTE: No GPU detected. Notebooks will run on CPU.")elif IN_COLAB and (TORCH_CUDA_AVAILABLE or TF_GPU_AVAILABLE):    print("\nGPU acceleration is available.")print("\nNotebook 01 complete - Environment ready for Project Aura 2.0")

In [ ]:
def get_version(module_name):
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "Unknown")
    except ImportError:
        return "Not installed"

print("\n" + "=" * 55)
print("PROJECT AURA 2.0 - ENVIRONMENT SUMMARY")
print("=" * 55)

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Production")
print("Python:", sys.version.split()[0])

print("\nAI FRAMEWORKS")
print("PyTorch:", TORCH_VERSION)
print("TensorFlow:", TENSORFLOW_VERSION)

print("\nGPU")
print("PyTorch CUDA:", TORCH_CUDA_AVAILABLE)
print("TensorFlow GPU:", TF_GPU_AVAILABLE)
print("GPU detected:", detect_gpu())
print("GPU enabled in config:", config.colab.gpu_enabled)

print("\nCOMPUTER VISION LIBRARIES")
print("NumPy:", get_version("numpy"))
print("Pillow:", get_version("PIL"))
print("OpenCV:", get_version("cv2"))
print("MediaPipe:", get_version("mediapipe"))
print("Ultralytics:", get_version("ultralytics"))
print("TorchVision:", get_version("torchvision"))

print("\nPROJECT")
print("Name:", PROJECT_NAME)
print("Version:", PROJECT_VERSION)
print("Directory:", PROJECT_DIR)
print("Python path configured:", PROJECT_DIR in sys.path)

print("=" * 55)

if IN_COLAB and GPU_AVAILABLE:
    print("SUCCESS: Colab GPU acceleration is available.")
elif IN_COLAB:
    print("WARNING: Colab detected, but no GPU is available.")
else:
    print("Environment check completed.")

In [ ]:
# ==========================================================================# Step 7: Optional - Mount Google Drive# ==========================================================================if os.getenv("MOUNT_GDRIVE", "false").lower() == "true" and IN_COLAB:    try:        from google.colab import drive        drive.mount('/content/drive')        print("Google Drive mounted at /content/drive")    except ImportError:        print("google.colab not available")else:    print("Google Drive mount skipped (set MOUNT_GDRIVE=true to enable)")

In [ ]:
import os

MOUNT_GDRIVE = False

if MOUNT_GDRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    print("Google Drive mounted successfully.")
else:
    print("Google Drive mounting skipped.")